# 1- Importing Libraries

In [ ]:
! pip install pandas
! pip install numpy
! pip install scikit-learn
! pip install nltk


In [102]:
import pandas as pd
import numpy as np
import re
from tqdm import tqdm
import nltk
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.naive_bayes import MultinomialNB
from nltk.tokenize import sent_tokenize,word_tokenize
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score,f1_score,classification_report,confusion_matrix
nltk.download("stopwords")
nltk.download("punkt_tab")
lemmatizer = WordNetLemmatizer()
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/prabhsandhu/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/prabhsandhu/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/prabhsandhu/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

# 2- Loading Dataset

In [104]:
df = pd.read_csv("Reviews.csv")
df = df.sample(200000)

# 3- Understanding Dataset

In [93]:
df.columns

Index(['Id', 'ProductId', 'UserId', 'ProfileName', 'HelpfulnessNumerator',
       'HelpfulnessDenominator', 'Score', 'Time', 'Summary', 'Text'],
      dtype='str')

In [31]:
df.shape

(200000, 10)

In [ ]:
df.nunique()
df["Score"].unique()

array([5, 1, 4, 2, 3])

In [ ]:
df['HelpfulnessNumerator']

0         1
1         0
2         1
3         3
4         0
         ..
568449    0
568450    0
568451    2
568452    1
568453    0
Name: HelpfulnessNumerator, Length: 568454, dtype: int64

# 4- Preprocessing the Dataset

In [105]:
df.isnull().sum()

Id                         0
ProductId                  0
UserId                     0
ProfileName                6
HelpfulnessNumerator       0
HelpfulnessDenominator     0
Score                      0
Time                       0
Summary                   12
Text                       0
dtype: int64

In [106]:
df.dropna(inplace=True)

In [96]:
df.duplicated().sum()

np.int64(0)

In [107]:
df.drop(['Id','UserId','Time','Summary','HelpfulnessNumerator','HelpfulnessDenominator','ProfileName'],axis=1,inplace=True)

In [108]:
df["Sentiment"] = df["Score"].apply(lambda x:"Negative" if x<=2 else ("Neutral" if x==3 else "Positive"))

In [109]:
df.drop("Score",axis=1,inplace=True)

# 5- Text Preprocessing Using NLP

In [110]:
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words("english"))  # load ONCE

text_preprocessed = []

for text in tqdm(df["Text"].astype(str)):  # ensure all rows are strings
    text = text.lower()
    text = re.sub(r'[^A-Za-z0-9]', ' ', text)
    
    words = word_tokenize(text)
    words = [lemmatizer.lemmatize(word) for word in words if word not in stop_words and len(word) >= 2]
    
    text_preprocessed.append(" ".join(words))

100%|██████████| 199982/199982 [01:08<00:00, 2932.42it/s]


# 6- Training Machine Learning Models

In [111]:
X=text_preprocessed
y = df["Sentiment"]

X_train, X_test,y_train,y_test = train_test_split(X,y,test_size=0.3,random_state=42)
pipe = Pipeline([
    ('tf-idf',TfidfVectorizer()),
    ('lgr',LogisticRegression(max_iter=1000))
])

pipe.fit(X_train,y_train)
y_pred = pipe.predict(X_test)

In [112]:
print(accuracy_score(y_test,y_pred)*100)
print(f1_score(y_test,y_pred,average="weighted")*100)

86.60888407367281
84.62722195981586
